# Multimodal text–image embeddings with vision language models

**Run in Google Colab only**.

**What you will learn**

- Load a **vision language model (VLM)** embedding model ([Qwen3-VL-Embedding-2B](https://huggingface.co/Qwen/Qwen3-VL-Embedding-2B)) with [Sentence Transformers](https://www.sbert.net/)
- Encode **text** and **image** inputs into one shared vector space
- Measure **cross-modal similarity** (text query vs image documents) and use `encode_query` / `encode_document` for retrieval-style APIs

**How to open (only supported path)**

1. [Google Colab](https://colab.research.google.com/) → **File → Open notebook → GitHub**
2. Enter URL: `https://github.com/ysskrishna/awesome-llm-experiments/blob/main/experiments/multimodal-text-image-vl-embeddings/notebook.ipynb`
3. **Runtime → Change runtime type → T4 GPU** (required; model needs ~8 GB VRAM)
4. Run all cells **top to bottom**

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ysskrishna/awesome-llm-experiments/blob/main/experiments/multimodal-text-image-vl-embeddings/notebook.ipynb)

**First run:** [`Qwen/Qwen3-VL-Embedding-2B`](https://huggingface.co/Qwen/Qwen3-VL-Embedding-2B) downloads from Hugging Face on first load (multi‑GB; exact size depends on format). Cached under `~/.cache/huggingface/hub/` in the Colab VM. Expect several minutes for install + download + first encode on a T4.

## Concepts (quick links)



| Idea | Link |
|------|------|
| Qwen3-VL embedding model | [Qwen3-VL-Embedding-2B](https://huggingface.co/Qwen/Qwen3-VL-Embedding-2B) |
| Sentence Transformers API | [Documentation](https://www.sbert.net/) |
| Modality gap (lower cross-modal scores) | See demo B below |

## 1. Colab environment — install dependencies and verify GPU

Installs `sentence-transformers` with the **image** extra (v5.4+ for multimodal encode). Fails fast if you are not in Colab or if no GPU is available.

In [1]:
# This notebook needs a Colab GPU (~8 GB VRAM). Local Jupyter is not supported.
try:
    import google.colab  # noqa: F401

    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if not IN_COLAB:
    raise RuntimeError(
        "This notebook is Colab-only. Open it from GitHub in Google Colab "
        "(File → Open notebook → GitHub) and enable a GPU runtime."
    )

In [2]:
%pip install -q -U "sentence-transformers[image]>=5.4"

In [3]:
import torch

# Qwen3-VL-Embedding-2B runs on GPU; CPU is too slow and may OOM.
if not torch.cuda.is_available():
    raise RuntimeError(
        "CUDA GPU not available. In Colab: Runtime → Change runtime type → "
        "Hardware accelerator → T4 GPU, then restart and run all cells."
    )

print(f"GPU: {torch.cuda.get_device_name(0)}")

GPU: Tesla T4


## 2. Define helpers and demo constants


In [4]:
# Vision-language model: maps text AND images into the same vector space.
MODEL_NAME = "Qwen/Qwen3-VL-Embedding-2B"

# Tiny demo corpus: two public images we will treat as "documents" to search over.
CAR_IMAGE = (
    "https://huggingface.co/datasets/huggingface/documentation-images/"
    "resolve/main/transformers/tasks/car.jpg"
)
BEE_IMAGE = (
    "https://huggingface.co/datasets/huggingface/documentation-images/"
    "resolve/main/bee.jpg"
)
DEMO_IMAGES = [CAR_IMAGE, BEE_IMAGE]  # index 0 = car, index 1 = bee
IMAGE_LABELS = ["car", "bee"]  # column names when printing similarity tables


def print_similarity_matrix(similarities, row_labels, col_labels):
    """Print text×image similarity scores with row/column labels."""
    # similarities[i, j] = how well text row i matches image column j (cosine-like score).
    sim = similarities.cpu()
    header = " " * 24 + "  ".join(f"{c:>8}" for c in col_labels)
    print(header)
    for i, row_name in enumerate(row_labels):
        scores = "  ".join(f"{sim[i, j].item():8.4f}" for j in range(sim.shape[1]))
        print(f"{row_name:24} {scores}")


def best_image_index(similarities, text_row: int, num_images: int) -> int:
    """Index of the image column with highest similarity for one text row."""
    # Retrieval rule: pick the image with the highest score for this text query.
    row = similarities[text_row, :num_images]
    return int(row.argmax().item())


def assert_text_prefers_image(similarities, text_row, expected_image_idx, caption):
    # We check ranking (correct image wins), not absolute score thresholds.
    best = best_image_index(similarities, text_row, similarities.shape[1])
    assert best == expected_image_idx, (
        f"{caption}: expected image index {expected_image_idx}, got {best}"
    )

## 3. Load the vision language embedding model

Downloads and loads **Qwen3-VL-Embedding-2B** on GPU. This cell is the slow step on first run.

In [5]:
from sentence_transformers import SentenceTransformer

# First run downloads ~4 GB from Hugging Face and caches under ~/.cache/huggingface/hub/.
model = SentenceTransformer(MODEL_NAME)

# This experiment needs image encoding; text-only models would fail here.
assert model.supports("image"), "Expected image modality support"
print(f"Modalities: {model.modalities}")  # text, image, video, message — we use text + image

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:112: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/429 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/238 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/24.2k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/770 [00:00<?, ?B/s]

...

added_tokens.json:   0%|          | 0.00/707 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/613 [00:00<?, ?B/s]

video_preprocessor_config.json:   0%|          | 0.00/817 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/96.0 [00:00<?, ?B/s]

Modalities: ['text', 'image', 'video', 'message']


## 4. Encode images — expect shape (2, embedding_dim)

Encodes two demo images from URLs. Embedding dimension is model-specific (2048 for this Qwen3-VL).

In [6]:
# encode() turns each image URL into a fixed-size vector (same space as text will use later).
# Shape: (num_images, embedding_dim) — here (2, 2048) for Qwen3-VL-Embedding-2B.
img_embeddings = model.encode(DEMO_IMAGES)
print(f"Image embeddings shape: {img_embeddings.shape}")
assert img_embeddings.shape[0] == len(DEMO_IMAGES)
assert img_embeddings.shape[1] > 0

Image embeddings shape: (2, 2048)


## 5. Cross-modal similarity — text queries vs image documents

Same car/bee example as the HF blog: two images, four text captions (match + hard negative per image). We assert **ranking** (which image wins), not absolute score thresholds.

**Modality gap:** cross-modal similarity scores are often lower than within-modal scores (e.g. text–text), but relative order still supports retrieval.

In [7]:
# --- Cross-modal similarity demo ---
# "Cross-modal" = compare different input types (text vs image) in one shared vector space.
# Rows = text queries; columns = image documents. Higher score = better semantic match.

texts = [
    "A green car parked in front of a yellow building",  # should match car image (idx 0)
    "A red car driving on a highway",  # car-related but harder (different scene)
    "A bee on a pink flower",  # should match bee image (idx 1)
    "A wasp on a wooden table",  # insect-related hard negative for bee image
]
text_labels = ["car_match", "car_negative", "bee_match", "bee_negative"]

# Step 1: embed images and texts into the same vector space.
img_embeddings = model.encode(DEMO_IMAGES)  # shape (2, 2048)
text_embeddings = model.encode(texts)  # shape (4, 2048)

# Step 2: pairwise scores — similarities[i, j] = text i vs image j.
# This is the core "search photos with words" operation (no vector DB yet).
similarities = model.similarity(text_embeddings, img_embeddings)

print_similarity_matrix(similarities, text_labels, IMAGE_LABELS)

# Step 3: verify retrieval ranking, not raw score magnitude (modality gap lowers scores).
assert_text_prefers_image(similarities, 0, 0, "Green car caption")
assert_text_prefers_image(similarities, 2, 1, "Bee on flower caption")
print("Rank checks passed: car caption → car image; bee caption → bee image.")

                             car       bee
car_match                  0.5117    0.1108
car_negative               0.2008    0.1139
bee_match                  0.1237    0.6788
bee_negative               0.1280    0.2727
Rank checks passed: car caption → car image; bee caption → bee image.


## 6. Retrieval-style API — `encode_query` and `encode_document`

Many retrieval models apply different prompts for queries vs documents. These methods wrap `encode()` with the model’s configured prompts when available.

In [8]:
# --- Retrieval-style API (closer to production search) ---
# Many embedders use different internal prompts for "I am searching" vs "I am being searched".
# encode_query = search side; encode_document = index side (images stored in a vector DB).

queries = [
    "Find me a photo of a vehicle parked near a building",  # expect car (idx 0)
    "Show me an image of a pollinating insect",  # expect bee (idx 1)
]

query_embeddings = model.encode_query(queries)  # user text at search time
doc_embeddings = model.encode_document(DEMO_IMAGES)  # images at index time
retrieval_sims = model.similarity(query_embeddings, doc_embeddings)

print_similarity_matrix(
    retrieval_sims,
    ["vehicle_query", "insect_query"],
    IMAGE_LABELS,
)

# Same idea as section 5: correct document should rank first for each query.
assert best_image_index(retrieval_sims, 0, 2) == 0, "Vehicle query should rank car image first"
assert best_image_index(retrieval_sims, 1, 2) == 1, "Insect query should rank bee image first"
print("Retrieval rank checks passed.")

                             car       bee
vehicle_query              0.3947    0.1515
insect_query               0.1266    0.4861
Retrieval rank checks passed.


## Wrap-up

You loaded a **multimodal VLM embedder**, encoded **text** and **images** into one space, and verified **cross-modal** ranking on a tiny demo corpus.

**If you hit OOM on Colab:** reload with lower memory, e.g. `SentenceTransformer(MODEL_NAME, model_kwargs={"torch_dtype": "bfloat16"})` (see the [blog’s processor/model kwargs section](https://huggingface.co/blog/multimodal-sentence-transformers#processor-and-model-kwargs)).

**Next steps (not covered here)**

- [Multimodal rerankers](https://huggingface.co/blog/multimodal-sentence-transformers#multimodal-reranker-models) (`CrossEncoder`) for higher-quality rescoring
- [Training multimodal embedders](https://huggingface.co/blog/train-multimodal-sentence-transformers)
- Index image/document embeddings in a vector DB for multimodal RAG